# HybridBid — T-60 Six-Method Comparison

**Eval window:** Jan 1 – Feb 23, 2026 (54 days, CT-aligned)  
**Battery:** 10 MW / 20 MWh, η=0.95 per-leg  
**Benchmark:** ERCOT fleet median $24.93/kW-yr; top-quartile $32.23/kW-yr  
**MILP-replay ceiling:** $58.40/kW-yr (continuous-SoC, 6D joint)  

This notebook loads `data/results/` outputs and produces the comparison table and plots for the professor meeting (Apr 30, 2026).

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

RESULTS_DIR = Path('../data/results')
FLEET_MEDIAN = 24.93
FLEET_TOP_Q  = 32.23
MILP_CEILING = 58.40

## 1. Load summary.json for each method

In [ ]:
METHOD_DIRS = {
    'TBx (baseline)':          'eval_tbx',
    'PF-MILP (oracle)':        'eval_pf_milp',
    'MILP+Forecaster':         'eval_milp_forecaster',
    'BC (offline)':            'eval_bc',
    'Cal-QL (offline)':        'eval_cal_ql',
    'Li et al. TempDRL':       'eval_tempdrl',
}

rows = []
for method_name, subdir in METHOD_DIRS.items():
    summary_path = RESULTS_DIR / subdir / 'summary.json'
    if not summary_path.exists():
        print(f'MISSING: {summary_path}')
        continue
    with open(summary_path) as f:
        d = json.load(f)
    rows.append({
        'Method': method_name,
        'All days ($/kW-yr)':    round(d['all_days_$_per_kW_yr'], 2),
        'ex-Fern ($/kW-yr)':    round(d['ex_fern_$_per_kW_yr'],  2),
        'Fern-only ($/kW)':     round(d['fern_only_$_per_kW'],   2),
        'Fleet percentile':      d['fleet_percentile'],
        'vs fleet median (%)':   round(d['vs_fleet_median'] * 100, 1),
        'vs MILP ceiling (%)':   round(d['vs_milp_replay_ceiling'] * 100, 1),
    })

df = pd.DataFrame(rows).set_index('Method')
df

## 2. Six-method comparison bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

colors = ['#4C72B0', '#4C72B0', '#55A868', '#C44E52', '#C44E52', '#8172B3']
bars = ax.barh(df.index[::-1], df['All days ($/kW-yr)'][::-1], color=colors[::-1], height=0.6)

ax.axvline(FLEET_MEDIAN, color='gray',   linestyle='--', linewidth=1.2, label=f'Fleet median ${FLEET_MEDIAN}/kW-yr')
ax.axvline(FLEET_TOP_Q,  color='orange', linestyle='--', linewidth=1.2, label=f'Fleet top-quartile ${FLEET_TOP_Q}/kW-yr')
ax.axvline(MILP_CEILING, color='black',  linestyle=':',  linewidth=1.2, label=f'MILP-replay ceiling ${MILP_CEILING}/kW-yr')

ax.set_xlabel('Annual revenue ($/kW-yr)', fontsize=12)
ax.set_title('HybridBid — T-60 Eval: All Days', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0f'))
ax.set_xlim(0, MILP_CEILING * 1.1)

for bar, val in zip(bars, df['All days ($/kW-yr)'][::-1]):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'${val:.1f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../data/results/comparison_all_days.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Three-way split (all / ex-Fern / Fern-only)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
splits = ['All days ($/kW-yr)', 'ex-Fern ($/kW-yr)', 'Fern-only ($/kW)']
titles = ['All days', 'Ex-Fern', 'Fern only']

for ax, col, title in zip(axes, splits, titles):
    vals = df[col]
    ax.barh(df.index[::-1], vals[::-1], color='steelblue', height=0.6)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('$/kW-yr' if 'yr' in col else '$/kW', fontsize=10)
    for i, v in enumerate(vals[::-1]):
        ax.text(v + 0.2, i, f'{v:.1f}', va='center', fontsize=8)

plt.suptitle('HybridBid — T-60 Revenue Split', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/results/comparison_three_way.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Fleet-percentile table

Reference: ERCOT fleet median \$24.93/kW-yr, top-quartile \$32.23/kW-yr (hub-average LMP proxy, same as this work).

In [ ]:
print(df[['All days ($/kW-yr)', 'Fleet percentile', 'vs fleet median (%)', 'vs MILP ceiling (%)']].to_string())